1.Write file program1_spark_streaming.py


In [ ]:
!pip install pyspark

from pyspark import SparkContext
from pyspark.streaming import StreamingContext

# Create Spark Context
sc = SparkContext.getOrCreate()
sc.setLogLevel("ERROR")

# Create Streaming Context
ssc = StreamingContext(sc, 5)

# Enable checkpointing
ssc.checkpoint("/content/checkpoint")

# Sample streaming data
data = [
    "hello spark",
    "spark streaming",
    "hello world"
]

# Create RDD Queue
rddQueue = [sc.parallelize(data) for i in range(5)]

# Create DStream
stream = ssc.queueStream(rddQueue)

# Split words
words = stream.flatMap(lambda x: x.split(" "))

# Count words
wordCounts = words.map(lambda x: (x, 1)) \
                  .reduceByKey(lambda a, b: a + b)

# Display results
wordCounts.pprint()

# Start streaming
ssc.start()

# Process for 30 seconds
ssc.awaitTerminationOrTimeout(30)

# Stop streaming
ssc.stop(stopSparkContext=True, stopGraceFully=True)

print("Streaming completed successfully!")

/usr/local/lib/python3.13/dist-packages/pyspark/streaming/context.py:72: FutureWarning: DStream is deprecated as of Spark 3.4.0. Migrate to Structured Streaming.
  warnings.warn(


-------------------------------------------
Time: 2026-11-16 12:51:25
-------------------------------------------
('hello', 2)
('streaming', 1)
('world', 1)
('spark', 2)

-------------------------------------------
Time: 2026-11-16 12:51:30
-------------------------------------------
('hello', 2)
('streaming', 1)
('world', 1)
('spark', 2)

-------------------------------------------
Time: 2026-11-16 12:51:35
-------------------------------------------
('hello', 2)
('streaming', 1)
('world', 1)
('spark', 2)

-------------------------------------------
Time: 2026-11-16 12:51:40
-------------------------------------------
('hello', 2)
('streaming', 1)
('world', 1)
('spark', 2)

-------------------------------------------
Time: 2026-11-16 12:51:45
-------------------------------------------
('hello', 2)
('streaming', 1)
('world', 1)
('spark', 2)

-------------------------------------------
Time: 2026-11-16 12:51:50
-------------------------------------------

------------------------------

2.Write file program2_duplicate_files.py

In [ ]:
from pyspark import SparkContext
from pyspark.streaming import StreamingContext
import hashlib

sc = SparkContext("local[2]", "DuplicateFileCheck")
sc.setLogLevel("ERROR")

ssc = StreamingContext(sc, 5)
ssc.checkpoint("/content/checkpoint")

# Sample file data
files = [
    ("file1.txt", "Hello Spark"),
    ("file2.txt", "Hello World"),
    ("file3.txt", "Hello Spark")
]

# Create streaming data
rddQueue = [sc.parallelize(files)]
stream = ssc.queueStream(rddQueue)

# Generate file hash
def get_hash(file):
    name, content = file
    return (hashlib.md5(content.encode()).hexdigest(), name)

# Find duplicate files
duplicates = stream.map(get_hash) \
    .groupByKey() \
    .filter(lambda x: len(list(x[1])) > 1) \
    .map(lambda x: (x[0], list(x[1])))

# Display results
duplicates.pprint()

ssc.start()
ssc.awaitTerminationOrTimeout(10)
ssc.stop(stopSparkContext=True, stopGraceFully=True)

-------------------------------------------
Time: 2026-11-16 12:41:20
-------------------------------------------
('b282bb63cae17b06c24af71b3c98abb1', ['file1.txt', 'file3.txt'])

-------------------------------------------
Time: 2026-11-16 12:41:25
-------------------------------------------

-------------------------------------------
Time: 2026-11-16 12:41:30
-------------------------------------------

-------------------------------------------
Time: 2026-11-16 12:41:35
-------------------------------------------

